## Paso 1 - Arranque y Stagging

In [7]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


spark = get_spark("novashop-m03")
orders = spark.read.parquet(str(STAGING / "orders_clean"))
items = spark.read.parquet(str(STAGING / "order_items_clean"))
print(orders.count(), items.count())


ROOT    /workspaces/python-pyspark-201
RAW     /workspaces/python-pyspark-201/data/raw existe: True
STAGING /workspaces/python-pyspark-201/data/staging
CURATED /workspaces/python-pyspark-201/data/curated
788 2010


## Paso 2 -- Inner para tener fecha



In [8]:
from pyspark.sql.functions import col

lines = items.join(orders, "order_id", "inner")
print(lines.count())

1980


# Paso 3 - GMV y mes

Si el varlor de discount es 1.50 el GMW es negativo

In [10]:
from pyspark.sql.functions import date_format

lines = (
    lines.withColumn(
        "gmv_line",
        col("qty") * col("unit_price") * (1 - col("discount")),
    ).withColumn("order_month", date_format(col("order_ts"), "yyyy-MM"))
)
lines.select("order_id", "qty", "unit_price", "discount", "gmv_line", "order_month").show(5)
print("gmv nulos", lines.where(col("gmv_line").isNull()).count())
print("gmv no nulos", lines.where(col("gmv_line").isNotNull()).count())
print("gmv < 0", lines.where(col("gmv_line") < 0).count())


+--------+---+----------+--------+--------+-----------+
|order_id|qty|unit_price|discount|gmv_line|order_month|
+--------+---+----------+--------+--------+-----------+
|  O00013|  2|    121.67|    0.00|243.3400|    2024-08|
|  O00013|  1|    128.00|    0.10|115.2000|    2024-08|
|  O00013|  5|    170.99|    0.10|769.4550|    2024-08|
|  O00013|  1|    237.56|    0.05|225.6820|    2024-08|
|  O00014|  2|    203.06|    0.00|406.1200|    2024-12|
+--------+---+----------+--------+--------+-----------+
only showing top 5 rows

gmv nulos 0
gmv no nulos 1980
gmv < 0 13


## Mejora

Cero nulos en gmv_line, en las 1980 files, 12 meses en 2024, 13 gmv negativos

In [20]:
print("gmv_nulos", lines.where(col("gmv_line").isNull()).count())
print("gmv negativos", lines.where(col("gmv_line") < 0).count())
print("meses de 2024", lines.where(col("order_month").substr(6,2) < "12").count())


gmv_nulos 0
gmv negativos 13
meses de 2024 1835


## Pedido de alto valor



En este caso el resultado es 1980 y debería sido 506. Revisar

In [23]:
lines.withColumn("is_high_value", col("gmv_line") >= 500)
print ("valores mayores de 500", lines.count())

lines.withColumn("is_high_value", col("gmv_line") >= 500).count()

valores mayores de 500 1980


1980

## lab 03 Reglas de negocio.

El notebook es autonomo y comienza de cero. Se podría poner entonces en otro fichero.

In [24]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col, date_format

spark = get_spark("novashop-m03")
orders = spark.read.parquet(str(STAGING / "orders_clean"))
items = spark.read.parquet(str(STAGING / "order_items_clean"))
lines = (
    items.join(orders, "order_id", "inner")
    .withColumn("gmv_line", col("qty") * col("unit_price") * (1 - col("discount")))
    .withColumn("order_month", date_format(col("order_ts"), "yyyy-MM"))
)
print(lines.count(), lines.where(col("gmv_line") < 0).count())

ROOT    /workspaces/python-pyspark-201
RAW     /workspaces/python-pyspark-201/data/raw existe: True
STAGING /workspaces/python-pyspark-201/data/staging
CURATED /workspaces/python-pyspark-201/data/curated
1980 13


# Paso 2 - Aplicar tres reglas y calcular gmw

In [25]:
from pyspark.sql.functions import when, lower, least, lit

fact = (
    lines.withColumn("discount", least(col("discount"), lit(1.0)))
    .withColumn(
        "channel_norm",
        when(lower(col("channel")).isin("web", "app", "store"), lower(col("channel")))
        .otherwise(lit("other")),
    )
    .withColumn("is_billable", col("status") == "paid")
    .withColumn(
        "gmv_line",
        col("qty") * col("unit_price") * (1 - col("discount")),
    )
)
print("filas", fact.count())
print("gmv < 0", fact.where(col("gmv_line") < 0).count())

filas 1980
gmv < 0 0


# Paso 3 - Valida el dominio



In [29]:
fact.groupBy("channel_norm").count().orderBy("channel_norm").show()
print("discount > 1", fact.where(col("discount") > 1).count())
print("billable", fact.where(col("is_billable")).count())

fact.show()

print("not billable", fact.where(col("is_billable")).count())

+------------+-----+
|channel_norm|count|
+------------+-----+
|         app|  715|
|       other|  247|
|       store|  243|
|         web|  775|
+------------+-----+

discount > 1 0
billable 1127
+--------+----------+---+----------+--------+-----------+---------+-----------+-------------------+--------+-----------+------------+-----------+
|order_id|product_id|qty|unit_price|discount|customer_id|   status|    channel|           order_ts|gmv_line|order_month|channel_norm|is_billable|
+--------+----------+---+----------+--------+-----------+---------+-----------+-------------------+--------+-----------+------------+-----------+
|  O00013|      P037|  2|    121.67|     0.0|      CX013|     paid|marketplace|2024-08-12 03:00:00|  243.34|    2024-08|       other|       true|
|  O00013|      P026|  1|    128.00|     0.1|      CX013|     paid|marketplace|2024-08-12 03:00:00|   115.2|    2024-08|       other|       true|
|  O00013|      P052|  5|    170.99|     0.1|      CX013|     paid|marke

# Paso 4 - Escribe fact-lines



In [30]:
dest = STAGING / "fact_lines"
fact.write.mode("overwrite").parquet(str(dest))
print(spark.read.parquet(str(dest)).count())


1980


In [31]:
fact.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- Project (7)
      +- BroadcastHashJoin Inner BuildRight (6)
         :- Filter (2)
         :  +- Scan parquet  (1)
         +- BroadcastExchange (5)
            +- Filter (4)
               +- Scan parquet  (3)


(1) Scan parquet 
Output [5]: [order_id#794, product_id#795, qty#796, unit_price#797, discount#798]
Batched: true
Location: InMemoryFileIndex [file:/workspaces/python-pyspark-201/data/staging/order_items_clean]
PushedFilters: [IsNotNull(order_id)]
ReadSchema: struct<order_id:string,product_id:string,qty:int,unit_price:decimal(10,2),discount:decimal(5,2)>

(2) Filter
Input [5]: [order_id#794, product_id#795, qty#796, unit_price#797, discount#798]
Condition : isnotnull(order_id#794)

(3) Scan parquet 
Output [5]: [order_id#784, customer_id#785, status#786, channel#787, order_ts#788]
Batched: true
Location: InMemoryFileIndex [file:/workspaces/python-pyspark-201/data/staging/orders_clean]
PushedFilters: [IsNotNull(ord